In [2]:
!pip install numpy 
# federated fuzzy means

import numpy as np

# Parameters
C = 2
m = 2.0
rounds = 20

# Simulated clients
client_data = [
    np.array([[1,1], [2,2], [3,3]], dtype=float),
    np.array([[8,8], [9,9], [10,10]], dtype=float)
]

# Initialize centroids
centroids = np.random.rand(C, 2)

def update_membership(data, centroids):
    N = data.shape[0]
    C = centroids.shape[0]
    U = np.zeros((N, C))

    for i in range(N):
        for j in range(C):
            d_ij = np.linalg.norm(data[i] - centroids[j])
            if d_ij == 0:
                U[i, :] = 0
                U[i, j] = 1
                continue

            sum_term = 0
            for k in range(C):
                d_ik = np.linalg.norm(data[i] - centroids[k])
                if d_ik == 0:
                    d_ik = 1e-6
                sum_term += (d_ij / d_ik) ** (2 / (m - 1))

            U[i, j] = 1 / sum_term
    return U

def compute_local_stats(data, U, centroids):
    um = U ** m

    # Local centroid
    c_local = (um.T @ data) / np.sum(um.T, axis=1, keepdims=True)

    # Weights for aggregation
    weights = np.sum(um, axis=0)

    # XB numerator contribution
    xb_local = 0
    for i in range(data.shape[0]):
        for j in range(C):
            dist_sq = np.linalg.norm(data[i] - centroids[j]) ** 2
            xb_local += um[i, j] * dist_sq

    return c_local, weights, xb_local, data.shape[0]

def compute_xb(centroids, xb_numerator, total_points):
    # separation (min distance between centroids)
    min_dist = float('inf')
    for i in range(C):
        for j in range(i + 1, C):
            dist = np.linalg.norm(centroids[i] - centroids[j]) ** 2
            if dist < min_dist:
                min_dist = dist

    return xb_numerator / (total_points * min_dist)

# Federated loop
for r in range(rounds):
    local_centroids = []
    weights = []
    xb_total = 0
    total_points = 0

    for data in client_data:
        U = update_membership(data, centroids)

        c_local, w_local, xb_local, n_local = compute_local_stats(
            data, U, centroids
        )

        local_centroids.append(c_local)
        weights.append(w_local)
        xb_total += xb_local
        total_points += n_local

    # Aggregate centroids
    new_centroids = np.zeros_like(centroids)
    total_weights = np.sum(weights, axis=0)

    for j in range(C):
        for k in range(len(client_data)):
            new_centroids[j] += weights[k][j] * local_centroids[k][j]
        new_centroids[j] /= total_weights[j]

    centroids = new_centroids

    # Compute XB index
    xb = compute_xb(centroids, xb_total, total_points)
    print(f"Round {r+1}, XB Index: {xb:.4f}")

print("\nFinal Centroids:\n", centroids)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 6.3 MB/s eta 0:00:0000:0100:01
Round 1, XB Index: 1.6779
Round 2, XB Index: 0.0823
Round 3, XB Index: 0.0146
Round 4, XB Index: 0.0133
Round 5, XB Index: 0.0133
Round 6, XB Index: 0.0133
Round 7, XB Index: 0.0133
Round 8, XB Index: 0.0133
Round 9, XB Index: 0.0133
Round 10, XB Index: 0.0133
Round 11, XB Index: 0.0133
Round 12, XB Index: 0.0133
Round 13, XB Index: 0.0133
Round 14, XB Index: 0.0133
Round 15, XB Index: 0.0133
Round 16, XB Index: 0.0133
Round 17, XB Index: 0.0133
Round 18, XB Index: 0.0133
Round 19, XB Index: 0.0133
Round 20, XB Index: 0.0133

Final Centroids:
 [[1.99403962 1.99403962]
 [9.00596038 9.00596038]]


In [ ]:
to get best k : use K=argminXB